### From scratch GNN baselines: GCN and GIN

Two message passing graph neural networks built directly with PyTorch
Geometric. Building GCN and GIN from scratch here makes it possible to
compare the effect of readout choice (mean vs sum pooling) and message
passing operator directly.

In [ ]:
from pathlib import Path
import pandas as pd
import wandb
from lightning import pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint
import numpy as np


In [ ]:
import torch
from torch_geometric.data import Data
from rdkit import Chem

### 1. Load scaffold split
Only the scaffold split is used for the GNN (not the random split contrast). Scaffold is the primary evaluation split.

In [ ]:
data_path = Path.cwd().parent
splits_dir = data_path / "data/splits"

In [ ]:
smiles_column = "smiles"
target_column = "pIC50"
train_df = pd.read_csv(splits_dir/"scaffold_train.csv")
val_df = pd.read_csv(splits_dir/"scaffold_val.csv")
test_df = pd.read_csv(splits_dir/"scaffold_test.csv")

train_df.shape, val_df.shape, test_df.shape

### 2. Build Molecule Datapoints, MoleculeDataset and DataLoader

In [ ]:
def atom_features(atom):
    """Per atom feature vector fed into the GNN as node features.

    Each atom becomes one row of the graph's node feature matrix. The
    fields below are a compact, commonly used set for molecular GNNs:
    element identity, local connectivity, and simple chemical properties
    that a message passing network can combine and propagate, rather than
    a bit vector fingerprint that is computed once and never updated.
    """
    return np.array([
        atom.GetAtomicNum(),          # element identity, e.g. 6 = carbon
        atom.GetDegree(),             # number of directly bonded neighbors
        atom.GetFormalCharge(),       # formal charge on the atom
        atom.GetHybridization().real, # orbital hybridization, e.g. sp2/sp3
        int(atom.GetIsAromatic()),    # part of an aromatic ring system
        atom.GetTotalNumHs(),         # implicit + explicit attached hydrogens
        int(atom.IsInRing()),         # part of any ring, aromatic or not
    ], dtype=np.float32)

def bond_features(bond):
    """Per bond feature vector, attached to each edge of the molecular graph.

    Bond type is encoded as a one hot vector (single/double/triple/aromatic)
    rather than a single ordinal value, since bond types are categorical,
    not ordered: there is no meaningful sense in which "double" sits
    between "single" and "triple".
    """
    bt = bond.GetBondType()
    return np.array([
        bt == Chem.rdchem.BondType.SINGLE,
        bt == Chem.rdchem.BondType.DOUBLE,
        bt == Chem.rdchem.BondType.TRIPLE,
        bt == Chem.rdchem.BondType.AROMATIC,
        int(bond.GetIsConjugated()),
        int(bond.IsInRing()),
    ], dtype=np.float32)

def smiles_to_graph(smiles, y):
    """Convert one SMILES string plus its target value into a PyG graph.

    Atoms become nodes, bonds become edges, and the pIC50 label is attached
    as the graph level target `y`. Returns None for SMILES RDKit can't
    parse, so invalid rows get silently dropped downstream instead of
    crashing the whole batch.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    x = torch.tensor(
        np.array([atom_features(atom) for atom in mol.GetAtoms()]),
        dtype=torch.float,
    )

    # Edges (both directions, since molecular graphs are undirected — if
    # atom i is bonded to atom j, message passing needs to send information
    # both i -> j and j -> i, so each bond becomes two directed edges).
    edge_index = []
    edge_attr = []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        feat = bond_features(bond)
        edge_index += [[i, j], [j, i]]
        edge_attr += [feat, feat]

    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    edge_attr = torch.tensor(np.array(edge_attr), dtype=torch.float)

    return Data(
        x=x,
        edge_index=edge_index,
        edge_attr=edge_attr,
        y=torch.tensor([y], dtype=torch.float),
    )

In [ ]:
from torch_geometric.data import InMemoryDataset

def build_graph_list(df, smiles_column, target_column):
    graphs = []
    for smi, y in zip(df[smiles_column], df[target_column]):
        g = smiles_to_graph(smi, y)
        if g is not None:
            graphs.append(g)
    return graphs

train_graphs = build_graph_list(train_df, "smiles", "pIC50")
val_graphs   = build_graph_list(val_df, "smiles", "pIC50")
test_graphs  = build_graph_list(test_df, "smiles", "pIC50")

# Counts should match the DataFrame row counts above — if they don't, some
# SMILES failed to parse and were dropped.
print(len(train_graphs), len(val_graphs), len(test_graphs))


In [ ]:
# Inspect one graph object's shape summary as a sanity check before training.
sample = train_graphs[8]
print(sample)

In [ ]:
# Print the raw tensors for that one molecule: one feature row per atom in
# x, a pair of directed edges per bond in edge_index/edge_attr (see the
# "both directions" comment above), and the single scalar pIC50 target y.
print("Atom features (x):\n", sample.x)
print("\nEdge index (connectivity):\n", sample.edge_index)
print("\nEdge attributes (bond features):\n", sample.edge_attr)
print("\nTarget (y):\n", sample.y)

In [ ]:
from sklearn.preprocessing import StandardScaler

# Standardize the target (zero mean, unit variance) before training. Neural
# nets trained with MSE loss are sensitive to the scale of the target.
train_y = np.array([g.y.item() for g in train_graphs]).reshape(-1, 1)
scaler = StandardScaler().fit(train_y)

def apply_scaling(graphs, scaler):
    for g in graphs:
        g.y = torch.tensor([scaler.transform([[g.y.item()]])[0, 0]], dtype=torch.float)
    return graphs

train_graphs = apply_scaling(train_graphs, scaler)
val_graphs   = apply_scaling(val_graphs, scaler)
# test_graphs stays on the original pIC50 scale — predictions on it get
# inverse transformed back to real pIC50 units before computing metrics, so
# RMSE/MAE are reported in interpretable pIC50 units, not standardized ones.

In [ ]:
# from torch_geometric.loader import DataLoader

train_loader = DataLoader(train_graphs, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_graphs, batch_size=64, shuffle=False)
test_loader  = DataLoader(test_graphs, batch_size=64, shuffle=False)


### GCN Model

Graph Convolutional Network: each layer aggregates (averages) neighboring
atom feature vectors, weighted by a normalized adjacency, then applies a
linear transform and nonlinearity. Stacking layers lets information from
progressively larger neighborhoods reach each atom.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool

class GCN(nn.Module):
    """Graph Convolutional Network regressor for pIC50.

    Stacks `num_layers` GCNConv message passing layers, each of which
    updates every atom's representation by aggregating a normalized average
    of its bonded neighbors' representations, then a graph level pooling
    step collapses all atom representations in a molecule into a single
    fixed size vector that a small MLP maps to one predicted pIC50 value.
    """
    def __init__(self, in_channels, hidden_channels=128, num_layers=4, dropout=0.2):
        super().__init__()
        self.convs = nn.ModuleList()
        self.batch_norms = nn.ModuleList()

        # num_layers message passing layers means information can travel up
        # to num_layers bonds away from each atom by the final layer —
        # e.g. 4 layers gives every atom a receptive field covering
        # everything within 4 bonds, roughly a small local substructure.
        for i in range(num_layers):
            in_dim = in_channels if i == 0 else hidden_channels
            self.convs.append(GCNConv(in_dim, hidden_channels))
            # BatchNorm after each conv stabilizes training by keeping
            # activation scales consistent across layers as depth grows.
            self.batch_norms.append(nn.BatchNorm1d(hidden_channels))

        self.dropout = dropout

        # Small MLP "readout" head maps the pooled graph embedding to a
        # single scalar prediction (the standardized pIC50 value).
        self.readout = nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_channels, 1),
        )

    def forward(self, x, edge_index, batch):
        for conv, bn in zip(self.convs, self.batch_norms):
            # Each GCNConv call is one round of message passing: every atom
            # collects a normalized average of its neighbors' features
            # (weighted by node degree), combines it with its own features,
            # and applies a learned linear transform.
            x = conv(x, edge_index)
            x = bn(x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)

        # Mean pooling here is standard for GCN
        # (GCN's own neighbor aggregation already uses a normalized average,
        # so mean pooling at readout is consistent with that design.)
        graph_embedding = global_mean_pool(x, batch)

        return self.readout(graph_embedding).squeeze(-1)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

in_channels = train_graphs[0].x.shape[1]
model = GCN(in_channels=in_channels).to(device)

# weight_decay applies L2 regularization to the optimizer, and MSE loss
# directly optimizes for RMSE (RMSE is just sqrt(MSE)), matching the
# regression metric this model is ultimately evaluated on.
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
criterion = nn.MSELoss()

def train_one_epoch(loader):
    model.train()
    total_loss = 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        pred = model(batch.x, batch.edge_index, batch.batch)
        loss = criterion(pred, batch.y)
        loss.backward()
        optimizer.step()
        # Weight by batch.num_graphs (not just the batch count) so the
        # final average is a true per molecule loss.
        total_loss += loss.item() * batch.num_graphs
    return total_loss / len(loader.dataset)

@torch.no_grad()
def evaluate(loader):
    model.eval()
    total_loss = 0
    for batch in loader:
        batch = batch.to(device)
        pred = model(batch.x, batch.edge_index, batch.batch)
        loss = criterion(pred, batch.y)
        total_loss += loss.item() * batch.num_graphs
    return total_loss / len(loader.dataset)

In [ ]:
best_val_loss = float("inf")
# Early stopping: if val loss hasn't improved for `patience` consecutive
# epochs, stop training. Prevents wasting compute (and overfitting further)
# once the model has stopped generalizing better.
patience, patience_counter = 15, 0
max_epochs = 200

run = wandb.init(
    project="egfr-pic50-prediction",
    job_type="gnn",
    group="scaffold",
    name="gcn_baseline_scaffold_split",
    config={
        "split_type": "scaffold",
        "descriptor_type": "learned_graph_GCN",
        "model": "GCN_mean_pooling",
        "num_layers": 4,
        "hidden_channels": 128,
        "dropout": 0.2,
        "lr": 1e-3,
        "train_size": len(train_graphs),
        "val_size": len(val_graphs),
        "test_size": len(test_graphs),
    },
)

for epoch in range(max_epochs):
    train_loss = train_one_epoch(train_loader)
    val_loss = evaluate(val_loader)

    if epoch % 10 == 0:
        print(f"Epoch {epoch}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")

    if val_loss < best_val_loss:
        # New best: reset patience and checkpoint these weights, so the
        # final saved model is whichever epoch generalized best to val, not
        # necessarily the last epoch trained.
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "models/gcn_best.pt")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

wandb.log({"best_val_loss": best_val_loss})
# Register the best checkpoint as a W&B model artifact for this run.
artifact = wandb.Artifact("gcn-scaffold", type="model")
artifact.add_file("models/gcn_best.pt")
run.log_artifact(artifact)
# run stays open — test metrics get logged to it in the next cell

In [ ]:
from sklearn.metrics import r2_score
from scipy.stats import spearmanr

# Reload the best checkpoint (lowest val loss during training), not
# whatever the final epoch's in-memory weights happen to be.
model.load_state_dict(torch.load("models/gcn_best.pt"))
model.eval()

all_preds = []
with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        pred = model(batch.x, batch.edge_index, batch.batch)
        all_preds.append(pred.cpu().numpy())

all_preds = np.concatenate(all_preds)
# Predictions were made in standardized target space (see the scaling cell
# above), so invert that transform to get predictions back in real pIC50
# units before computing metrics against the unscaled test labels.
all_preds_unscaled = scaler.inverse_transform(all_preds.reshape(-1, 1)).flatten()
all_true = test_df["pIC50"].values

rmse = np.sqrt(np.mean((all_preds_unscaled - all_true) ** 2))
mae = np.mean(np.abs(all_preds_unscaled - all_true))
r2 = r2_score(all_true, all_preds_unscaled)
spearman_rho, _ = spearmanr(all_true, all_preds_unscaled)

# Spearman rho (rank correlation) matters alongside RMSE/MAE/R2 in drug
# discovery, since ranking candidate molecules correctly often matters more
# than getting the exact potency value right.
print(f"GCN Test RMSE: {rmse:.4f}, MAE: {mae:.4f}, R²: {r2:.4f}, Spearman ρ: {spearman_rho:.4f}")

wandb.log({
    "test_rmse": rmse, "test_mae": mae,
    "test_r2": r2, "test_spearman_rho": spearman_rho,
})
wandb.finish()

### GIN model

Graph Isomorphism Network: theoretically the most expressive message
passing GNN variant (under the Weisfeiler-Lehman graph isomorphism test),
because it uses sum aggregation and an MLP instead of GCN's normalized mean
aggregation. Sum preserves more information about neighbor multiplicities
than mean does, which matters for distinguishing structurally different
molecules that GCN's averaging could otherwise conflate.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GINConv, global_add_pool

class GIN(nn.Module):
    """Graph Isomorphism Network regressor for pIC50.

    Same overall shape as the GCN above (stacked message passing layers,
    batch norm, dropout, then a pooled readout MLP), but GINConv aggregates
    neighbor features by summation and passes the result through a small
    MLP, rather than GCN's normalized mean aggregation. Summing preserves
    information about how many neighbors of each kind an atom has, which
    the GIN paper shows makes this layer as expressive as the
    Weisfeiler-Lehman graph isomorphism test at distinguishing non
    isomorphic graphs.
    """
    def __init__(self, in_channels, hidden_channels=128, num_layers=4, dropout=0.2):
        super().__init__()
        self.convs = nn.ModuleList()
        self.batch_norms = nn.ModuleList()

        for i in range(num_layers):
            in_dim = in_channels if i == 0 else hidden_channels
            # GINConv's update rule is MLP((1+eps) * x_i + sum of neighbor
            # x_j). The MLP here is what gives GIN its extra representational
            # power over GCN's plain linear transform.
            mlp = nn.Sequential(
                nn.Linear(in_dim, hidden_channels),
                nn.ReLU(),
                nn.Linear(hidden_channels, hidden_channels),
            )
            # train_eps=True lets the model learn the (1+ε) weighting itself
            self.convs.append(GINConv(mlp, train_eps=True))
            self.batch_norms.append(nn.BatchNorm1d(hidden_channels))

        self.dropout = dropout

        self.readout = nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_channels, 1),
        )

    def forward(self, x, edge_index, batch):
        for conv, bn in zip(self.convs, self.batch_norms):
            x = conv(x, edge_index)
            x = bn(x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)

        # Sum pooling over all atoms in each molecule -> one vector per molecule
        # (sum, not mean, for the same injectivity reason GIN uses sum internally)
        graph_embedding = global_add_pool(x, batch)

        return self.readout(graph_embedding).squeeze(-1)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

in_channels = train_graphs[0].x.shape[1]
model = GIN(in_channels=in_channels).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
criterion = nn.MSELoss()

def train_one_epoch(loader):
    model.train()
    total_loss = 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        pred = model(batch.x, batch.edge_index, batch.batch)
        loss = criterion(pred, batch.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch.num_graphs
    return total_loss / len(loader.dataset)

@torch.no_grad()
def evaluate(loader):
    model.eval()
    total_loss = 0
    for batch in loader:
        batch = batch.to(device)
        pred = model(batch.x, batch.edge_index, batch.batch)
        loss = criterion(pred, batch.y)
        total_loss += loss.item() * batch.num_graphs
    return total_loss / len(loader.dataset)

In [ ]:
best_val_loss = float("inf")
patience, patience_counter = 15, 0
max_epochs = 200

run = wandb.init(
    project="egfr-pic50-prediction",
    job_type="gnn",
    group="scaffold",
    name="gin_baseline_scaffold_split",
    config={
        "split_type": "scaffold",
        "descriptor_type": "learned_graph_GIN",
        "model": "GIN_sum_pooling",
        "num_layers": 4,
        "hidden_channels": 128,
        "dropout": 0.2,
        "lr": 1e-3,
        "train_size": len(train_graphs),
        "val_size": len(val_graphs),
        "test_size": len(test_graphs),
    },
)

for epoch in range(max_epochs):
    train_loss = train_one_epoch(train_loader)
    val_loss = evaluate(val_loader)

    if epoch % 10 == 0:
        print(f"Epoch {epoch}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "models/gin_best.pt")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

wandb.log({"best_val_loss": best_val_loss})
artifact = wandb.Artifact("gin-scaffold", type="model")
artifact.add_file("models/gin_best.pt")
run.log_artifact(artifact)
wandb.finish()

In [ ]:
from sklearn.metrics import r2_score
from scipy.stats import spearmanr

model.load_state_dict(torch.load("models/gin_best.pt"))
model.eval()

all_preds, all_true = [], []
with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        pred = model(batch.x, batch.edge_index, batch.batch)
        all_preds.append(pred.cpu().numpy())
    
all_preds = np.concatenate(all_preds)

all_preds_unscaled = scaler.inverse_transform(all_preds.reshape(-1, 1)).flatten()
all_true = test_df["pIC50"].values

rmse = np.sqrt(np.mean((all_preds_unscaled - all_true) ** 2))
mae = np.mean(np.abs(all_preds_unscaled - all_true))
r2 = r2_score(all_true, all_preds_unscaled)
spearman_rho, _ = spearmanr(all_true, all_preds_unscaled)

print(f"Test RMSE: {rmse:.4f}, MAE: {mae:.4f}, R²: {r2:.4f}, Spearman ρ: {spearman_rho:.4f}")